# Tentative eyewear model comparison (VGG16 vs glasses-detector)

**Status:** This notebook is prepared but **not run yet**. It assumes comparison CSVs already exist from `evaluate_glasses_detector.py` (also tentative / not run yet).

Compare legacy VGG16 eyewear scores against `glasses-detector==0.1.1` (`AnyglassesClassifier`, small) on the same stored test images used in `All_Models_Evaluation.ipynb`.

## Background: limitations of VGG16 eyewear detection

The production backend (`detect_shades`) uses ImageNet VGG16 top-5 labels and sets eyewear to true when the substring `"sunglasses"` appears in the combined output for `subject_reference.png` and `face_tight_crop.png`.

Structural limits:

1. **Wrong task** — ImageNet classifies general objects, not "person wearing glasses." There is no reliable clear-eyeglasses class; only labels like `sunglasses` / `sunglass` and unrelated `field glasses`.
2. **Sunglasses-only trigger** — regular prescription glasses rarely appear as `sunglasses` in top-5, so `sunglasses_glasses.csv` is mostly 0% despite visible glasses (legacy accuracy ~22.5%).
3. **False positives** — unrelated top-5 labels (wig, mask, etc.) can occasionally include `sunglasses`; the code uses a string match with no score threshold.
4. **Out-of-domain crops** — tight face PNGs are noisy inputs for scene-object ImageNet classifiers.

## Why glasses-detector is being considered

`glasses-detector==0.1.x` provides pretrained **face-attribute** classifiers. `AnyglassesClassifier` (small) combines eyeglasses + sunglasses detectors and outputs a **probability** (0–1) instead of searching label text. It targets the UI question **"any eyewear?"** more directly than VGG16.

**Detection rule (same as legacy notebook):** treat `Predict (%) > 0` as a positive detection.

## Inputs (pre-exported, tentative)

Run from `testing/` after Python 3.10+ setup:

```bash
pip install glasses-detector==0.1.1 pandas
python evaluate_glasses_detector.py
```

Place PNGs under `testing/images/`. Expected exports:

- `Stored Test Results/sunglasses_shades_glasses_detector.csv`
- `Stored Test Results/sunglasses_glasses_glasses_detector.csv`

Columns: `Name`, `Size (KB)`, `VGG16`, `glasses-detector`

In [ ]:
import pandas as pd

SHADES_COMPARE_CSV = "Stored Test Results/sunglasses_shades_glasses_detector.csv"
GLASSES_COMPARE_CSV = "Stored Test Results/sunglasses_glasses_glasses_detector.csv"

shades_compare_df = pd.read_csv(SHADES_COMPARE_CSV, header=0)
glasses_compare_df = pd.read_csv(GLASSES_COMPARE_CSV, header=0)

def detection_accuracy(df, column, total):
    return round((df[column] > 0).sum() / total * 100, 2)

shades_compare_df.head()

## Sunglasses test set

Testing set shown below to detect sunglasses, as VGG16 originally intended:

<div>
<img src="Figures/Sunglasses_dataset.png" width="800"/>
</div>

In [ ]:
shades_compare_df

In [ ]:
shades_total = 20
shades_vgg16_accuracy = detection_accuracy(shades_compare_df, "VGG16", shades_total)
shades_gd_accuracy = detection_accuracy(shades_compare_df, "glasses-detector", shades_total)

print("Sunglasses set — VGG16 accuracy:", shades_vgg16_accuracy, "%")
print("Sunglasses set — glasses-detector accuracy:", shades_gd_accuracy, "%")

## Regular glasses test set

This set checks whether the model detects **clear / prescription glasses** (not only sunglasses):

<div>
<img src="Figures/Glasses_dataset.png" width="800"/>
</div>

In [ ]:
glasses_compare_df

In [ ]:
glasses_total = 40
glasses_vgg16_accuracy = detection_accuracy(glasses_compare_df, "VGG16", glasses_total)
glasses_gd_accuracy = detection_accuracy(glasses_compare_df, "glasses-detector", glasses_total)

print("Regular glasses set — VGG16 accuracy:", glasses_vgg16_accuracy, "%")
print("Regular glasses set — glasses-detector accuracy:", glasses_gd_accuracy, "%")

## Summary

Side-by-side detection rate (`Predict > 0`) for both models on each test set. Legacy VGG16 sunglasses accuracy on the regular-glasses set was **22.5%** in `All_Models_Evaluation.ipynb`.

In [ ]:
summary_df = pd.DataFrame(
    {
        "Test set": ["Sunglasses (n=20)", "Regular glasses (n=40)"],
        "VGG16 (%)": [shades_vgg16_accuracy, glasses_vgg16_accuracy],
        "glasses-detector (%)": [shades_gd_accuracy, glasses_gd_accuracy],
    }
)
summary_df

### Optional: per-image disagreement

Rows where one model detected eyewear and the other did not (`> 0` vs `0`).

In [ ]:
def disagreement_rows(df):
    vgg16_pos = df["VGG16"] > 0
    gd_pos = df["glasses-detector"] > 0
    return df[vgg16_pos ^ gd_pos][["Name", "VGG16", "glasses-detector"]]

print("Sunglasses set disagreements:")
display(disagreement_rows(shades_compare_df))

print("Regular glasses set disagreements:")
display(disagreement_rows(glasses_compare_df))